# Day 8 – Temporal and Sensor Data-Quality Audit

## Objective

This notebook audits the time-stamped water-quality dataset before modeling.

The audit focuses on:

- Timestamp validity and ordering
- Duplicate timestamps and duplicate records
- Missing time intervals and data gaps
- Sampling-frequency consistency
- Irregular sampling
- Missing sensor measurements
- Invalid or physically implausible sensor ranges
- Sensor-wise data quality
- Possible sensor drift
- Data leakage risks

The raw dataset is preserved without modification. All cleaning and resampling decisions are documented before creating a processed dataset.

In [2]:
import pandas as pd
import numpy as np

raw_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\raw\water_potability.csv"

df = pd.read_csv(raw_path)

print("Original dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

Original dataset shape: (3276, 10)
Columns:
['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability']


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


In [3]:
# Step 2: Generate synthetic timestamps

np.random.seed(42)

n = len(df)

# Start date for the synthetic timeline
start_time = pd.Timestamp("2025-01-01 00:00:00")

# Default sampling interval = 1 hour
intervals = np.full(n - 1, 60, dtype=int)  # minutes

# Introduce irregular sampling
irregular_positions = np.random.choice(
    np.arange(n - 1),
    size=min(30, n - 1),
    replace=False
)

irregular_values = np.random.choice(
    [15, 30, 90, 120],
    size=len(irregular_positions)
)

intervals[irregular_positions] = irregular_values

# Introduce larger missing intervals (gaps)
gap_positions = [500, 1200, 2000]

for position in gap_positions:
    if position < len(intervals):
        intervals[position] = 240  # 4-hour gap

# Convert intervals into cumulative timestamps
timestamps = [start_time]

for minutes in intervals:
    timestamps.append(
        timestamps[-1] + pd.Timedelta(minutes=int(minutes))
    )

# Create a new dataframe
timestamped_df = df.copy()
timestamped_df["timestamp"] = timestamps

# Put timestamp as the first column
timestamped_df = timestamped_df[
    ["timestamp"] + [col for col in timestamped_df.columns if col != "timestamp"]
]

print("Synthetic timestamped dataset created.")
print("Number of records:", len(timestamped_df))

display(timestamped_df.head(10))

Synthetic timestamped dataset created.
Number of records: 3276


,timestamp,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,2025-01-01 00:00:00,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,2025-01-01 01:00:00,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,2025-01-01 02:00:00,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,2025-01-01 03:00:00,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,2025-01-01 04:00:00,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0
5,2025-01-01 05:00:00,5.584087,188.313324,28748.687739,7.544869,326.678363,280.467916,8.399735,54.917862,2.559708,0
6,2025-01-01 06:00:00,10.223862,248.071735,28749.716544,7.513408,393.663396,283.651634,13.789695,84.603556,2.672989,0
7,2025-01-01 07:00:00,8.635849,203.361523,13672.091764,4.563009,303.309771,474.607645,12.363817,62.798309,4.401425,0
8,2025-01-01 08:00:00,NaN,118.988579,14285.583854,7.804174,268.646941,389.375566,12.706049,53.928846,3.595017,0
9,2025-01-01 09:00:00,11.180284,227.231469,25484.508491,9.077200,404.041635,563.885481,17.927806,71.976601,4.370562,0


In [4]:
# Step 3: Introduce controlled duplicate timestamps

timestamped_df = timestamped_df.copy()

# Create a few duplicate timestamps for data-quality testing
duplicate_positions = [100, 1000]

for position in duplicate_positions:
    if position < len(timestamped_df):
        timestamped_df.loc[position, "timestamp"] = (
            timestamped_df.loc[position - 1, "timestamp"]
        )

print("Controlled duplicate timestamps introduced.")

print(
    "Number of duplicate timestamps:",
    timestamped_df["timestamp"].duplicated().sum()
)

display(timestamped_df.iloc[[99, 100, 101]])

Controlled duplicate timestamps introduced.
Number of duplicate timestamps: 2


,timestamp,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
99,2025-01-05 02:30:00,9.177870,163.274828,20868.331219,7.726040,320.421432,426.994393,10.214275,62.430926,3.108770,0
100,2025-01-05 02:30:00,4.270716,166.837709,24405.250535,7.391510,297.206739,349.518910,13.478526,54.474840,3.763906,0
101,2025-01-05 04:30:00,7.348845,223.264940,11519.046527,8.889559,NaN,494.158434,18.145374,73.715522,4.880875,0


In [5]:
# Step 4: Save processed timestamped dataset

output_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed\water_quality_timestamped.csv"

timestamped_df.to_csv(output_path, index=False)

print("Processed dataset saved successfully!")
print("Location:", output_path)
print("Shape:", timestamped_df.shape)

Processed dataset saved successfully!
Location: C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed\water_quality_timestamped.csv
Shape: (3276, 11)


In [6]:
# Step 5: Verify saved processed dataset

check_df = pd.read_csv(output_path)

check_df["timestamp"] = pd.to_datetime(check_df["timestamp"])

print("Processed dataset loaded successfully.")
print("Rows:", len(check_df))
print("Columns:", len(check_df.columns))
print("Start timestamp:", check_df["timestamp"].min())
print("End timestamp:", check_df["timestamp"].max())
print("Duplicate timestamps:", check_df["timestamp"].duplicated().sum())

display(check_df.head())

Processed dataset loaded successfully.
Rows: 3276
Columns: 11
Start timestamp: 2025-01-01 00:00:00
End timestamp: 2025-05-17 23:45:00
Duplicate timestamps: 2


,timestamp,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,2025-01-01 00:00:00,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,2025-01-01 01:00:00,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,2025-01-01 02:00:00,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,2025-01-01 03:00:00,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,2025-01-01 04:00:00,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


In [7]:
# Step 6: Load processed temporal dataset

processed_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed\water_quality_timestamped.csv"

df_time = pd.read_csv(processed_path)

df_time["timestamp"] = pd.to_datetime(
    df_time["timestamp"],
    errors="coerce"
)

print("Dataset shape:", df_time.shape)
display(df_time.head())

Dataset shape: (3276, 11)


,timestamp,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,2025-01-01 00:00:00,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,2025-01-01 01:00:00,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,2025-01-01 02:00:00,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,2025-01-01 03:00:00,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,2025-01-01 04:00:00,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


In [8]:
# Step 7: Timestamp quality audit

print("Invalid timestamps:", df_time["timestamp"].isna().sum())
print("Duplicate timestamps:", df_time["timestamp"].duplicated().sum())
print("Duplicate complete records:", df_time.duplicated().sum())
print("Chronologically sorted:", df_time["timestamp"].is_monotonic_increasing)

print("\nStart:", df_time["timestamp"].min())
print("End:", df_time["timestamp"].max())

Invalid timestamps: 0
Duplicate timestamps: 2
Duplicate complete records: 0
Chronologically sorted: True

Start: 2025-01-01 00:00:00
End: 2025-05-17 23:45:00


In [9]:
# Step 8: Sampling interval analysis

df_time = df_time.sort_values("timestamp").reset_index(drop=True)

df_time["sampling_interval_minutes"] = (
    df_time["timestamp"].diff().dt.total_seconds() / 60
)

print("Sampling interval distribution:")
display(
    df_time["sampling_interval_minutes"]
    .value_counts()
    .sort_index()
)

Sampling interval distribution:


sampling_interval_minutes
0.0         2
15.0        7
30.0        7
60.0     3238
90.0        7
120.0      11
240.0       3
Name: count, dtype: int64

In [10]:
# Step 9: Identify large temporal gaps

expected_interval = 60  # expected sampling interval = 1 hour

gap_threshold = expected_interval * 1.5

large_gaps = df_time[
    df_time["sampling_interval_minutes"] > gap_threshold
]

print("Number of large gaps:", len(large_gaps))

display(
    large_gaps[
        ["timestamp", "sampling_interval_minutes"]
    ].head(20)
)

Number of large gaps: 14


,timestamp,sampling_interval_minutes
101,2025-01-05 04:30:00,120.0
450,2025-01-19 19:15:00,120.0
480,2025-01-21 02:45:00,120.0
501,2025-01-22 03:15:00,240.0
666,2025-01-29 01:00:00,120.0
804,2025-02-03 20:00:00,120.0
1001,2025-02-12 01:00:00,120.0
1201,2025-02-20 12:00:00,240.0
1504,2025-03-05 02:00:00,120.0
2001,2025-03-25 21:15:00,240.0


In [11]:
# Step 10: Identify irregular sampling

irregular_sampling = df_time[
    (df_time["sampling_interval_minutes"] != expected_interval)
    & (df_time["sampling_interval_minutes"] != 0)
]

print("Number of irregular intervals:", len(irregular_sampling))

display(
    irregular_sampling[
        ["timestamp", "sampling_interval_minutes"]
    ].head(20)
)

Number of irregular intervals: 36


,timestamp,sampling_interval_minutes
0,2025-01-01 00:00:00,NaN
94,2025-01-04 21:30:00,30.0
101,2025-01-05 04:30:00,120.0
140,2025-01-06 20:00:00,90.0
239,2025-01-10 22:15:00,15.0
290,2025-01-13 01:45:00,90.0
292,2025-01-13 04:15:00,90.0
450,2025-01-19 19:15:00,120.0
464,2025-01-20 09:45:00,90.0
480,2025-01-21 02:45:00,120.0


In [12]:
# Step 11: Parameter-wise data-quality table

# Exclude timestamp and sampling interval columns
parameter_columns = [
    col for col in df_time.columns
    if col not in ["timestamp", "sampling_interval_minutes"]
]

quality_results = []

for column in parameter_columns:
    series = df_time[column]

    quality_results.append({
        "Parameter": column,
        "Data Type": str(series.dtype),
        "Total Records": len(series),
        "Missing Values": series.isna().sum(),
        "Missing %": round(series.isna().mean() * 100, 2),
        "Unique Values": series.nunique(),
        "Minimum": series.min(),
        "Maximum": series.max(),
        "Mean": round(series.mean(), 2),
        "Std Dev": round(series.std(), 2)
    })

quality_table = pd.DataFrame(quality_results)

display(quality_table)

,Parameter,Data Type,Total Records,Missing Values,Missing %,Unique Values,Minimum,Maximum,Mean,Std Dev
0,ph,float64,3276,491,14.99,2785,0.000000,14.000000,7.08,1.59
1,Hardness,float64,3276,0,0.00,3276,47.432000,323.124000,196.37,32.88
2,Solids,float64,3276,0,0.00,3276,320.942611,61227.196008,22014.09,8768.57
3,Chloramines,float64,3276,0,0.00,3276,0.352000,13.127000,7.12,1.58
4,Sulfate,float64,3276,781,23.84,2495,129.000000,481.030642,333.78,41.42
5,Conductivity,float64,3276,0,0.00,3276,181.483754,753.342620,426.21,80.82
6,Organic_carbon,float64,3276,0,0.00,3276,2.200000,28.300000,14.28,3.31
7,Trihalomethanes,float64,3276,162,4.95,3114,0.738000,124.000000,66.40,16.18
8,Turbidity,float64,3276,0,0.00,3276,1.450000,6.739000,3.97,0.78
9,Potability,int64,3276,0,0.00,2,0.000000,1.000000,0.39,0.49


In [13]:
# Step 12: Check invalid parameter values

range_rules = {
    "ph": (0, 14),
    "Hardness": (0, None),
    "Solids": (0, None),
    "Chloramines": (0, None),
    "Sulfate": (0, None),
    "Conductivity": (0, None),
    "Organic_carbon": (0, None),
    "Trihalomethanes": (0, None),
    "Turbidity": (0, None),
    "Potability": (0, 1)
}

invalid_results = []

for column, (lower, upper) in range_rules.items():

    if column not in df_time.columns:
        continue

    series = pd.to_numeric(df_time[column], errors="coerce")

    if upper is None:
        invalid_count = (series < lower).sum()
    else:
        invalid_count = ((series < lower) | (series > upper)).sum()

    invalid_results.append({
        "Parameter": column,
        "Invalid Values": int(invalid_count),
        "Valid Values": int(series.notna().sum() - invalid_count)
    })

invalid_table = pd.DataFrame(invalid_results)

display(invalid_table)

,Parameter,Invalid Values,Valid Values
0,ph,0,2785
1,Hardness,0,3276
2,Solids,0,3276
3,Chloramines,0,3276
4,Sulfate,0,2495
5,Conductivity,0,3276
6,Organic_carbon,0,3276
7,Trihalomethanes,0,3114
8,Turbidity,0,3276
9,Potability,0,3276


In [14]:
# Step 13: Missing-value audit

missing_table = pd.DataFrame({
    "Parameter": df_time.columns,
    "Missing Values": df_time.isna().sum(),
    "Missing %": (df_time.isna().mean() * 100).round(2)
})

display(missing_table)

,Parameter,Missing Values,Missing %
timestamp,timestamp,0,0.00
ph,ph,491,14.99
Hardness,Hardness,0,0.00
Solids,Solids,0,0.00
Chloramines,Chloramines,0,0.00
Sulfate,Sulfate,781,23.84
Conductivity,Conductivity,0,0.00
Organic_carbon,Organic_carbon,0,0.00
Trihalomethanes,Trihalomethanes,162,4.95
Turbidity,Turbidity,0,0.00


In [15]:
# Step 14: Simple temporal drift check

numeric_columns = [
    col for col in df_time.select_dtypes(include=np.number).columns
    if col != "sampling_interval_minutes"
]

midpoint = len(df_time) // 2

drift_results = []

for column in numeric_columns:

    first_half_mean = df_time.iloc[:midpoint][column].mean()
    second_half_mean = df_time.iloc[midpoint:][column].mean()

    drift_results.append({
        "Parameter": column,
        "First Half Mean": round(first_half_mean, 2),
        "Second Half Mean": round(second_half_mean, 2),
        "Mean Difference": round(
            second_half_mean - first_half_mean, 2
        )
    })

drift_table = pd.DataFrame(drift_results)

display(drift_table)

,Parameter,First Half Mean,Second Half Mean,Mean Difference
0,ph,7.08,7.08,-0.00
1,Hardness,196.96,195.78,-1.18
2,Solids,21929.69,22098.50,168.81
3,Chloramines,7.10,7.15,0.05
4,Sulfate,331.50,336.05,4.55
5,Conductivity,426.92,425.49,-1.43
6,Organic_carbon,14.23,14.34,0.12
7,Trihalomethanes,66.82,65.98,-0.84
8,Turbidity,3.95,3.98,0.03
9,Potability,0.39,0.39,0.00


# Cleaning and Resampling Plan

## 1. Raw Data Preservation

The original water-quality dataset will be preserved without modification.

All cleaning and transformation operations will be performed on a separate working/processed copy.

## 2. Timestamp Handling

* Convert timestamps to a standard datetime format.
* Sort records chronologically.
* Flag invalid or missing timestamps.
* Identify duplicate timestamps.
* Duplicate timestamps will be investigated before removal because multiple observations at the same timestamp may represent repeated measurements.

## 3. Missing Time Intervals

* The dominant sampling interval will be identified from the timestamp differences.
* Intervals substantially larger than the expected sampling interval will be flagged as temporal gaps.
* Short gaps may be considered for interpolation during later modeling experiments.
* Long gaps will not be blindly filled because artificial values may introduce misleading temporal patterns.
* Gap indicators will be retained as metadata where appropriate.

## 4. Irregular Sampling

* Sampling intervals different from the expected frequency will be flagged.
* The project will use a consistent sampling frequency for time-series modeling where required.
* Resampling will be performed only after understanding the original sampling distribution.

## 5. Missing Parameter Values

* Missing values will first be quantified for each water-quality parameter.
* For short temporal gaps, interpolation methods may be evaluated.
* For longer missing periods, alternative methods such as forward filling, statistical imputation, or leaving values missing will be compared depending on the parameter and modeling task.
* Imputation decisions will be documented and validated.

## 6. Invalid Parameter Values

* Values outside physically or logically valid ranges will be flagged.
* pH values will be checked against the 0–14 range.
* Negative values for measured water-quality quantities will be flagged for investigation.
* Potability values will be checked to ensure they contain only the expected binary classes.

## 7. Duplicate Records

* Exact duplicate records will be identified.
* Exact duplicates may be removed during the cleaning stage.
* Duplicate timestamps with different measurements will not automatically be removed; they will first be investigated.

## 8. Data Leakage Prevention

To prevent temporal leakage:

* Data will be split chronologically rather than randomly.
* Future observations will not be used to create features for past observations.
* Interpolation and imputation will be fitted only using information available within the appropriate training period.
* Rolling statistics will use past observations only.
* Target variables and future values will not be used as input features.
* Synthetic quality-control flags will not be used as model features.

## 9. Resampling Strategy

The dominant sampling interval will be used as the initial candidate frequency.

If resampling is required:

1. Sort data by timestamp.
2. Set timestamp as the temporal index.
3. Resample to the selected frequency.
4. Aggregate or interpolate parameters according to their characteristics.
5. Preserve missingness/gap indicators.
6. Validate the resulting dataset before modeling.

## 10. Validation After Cleaning

After cleaning and resampling, the following checks will be repeated:

* Timestamp validity
* Chronological ordering
* Duplicate timestamps
* Sampling interval consistency
* Missing values
* Invalid parameter values
* Number and duration of remaining gaps
* Dataset size before and after processing

The raw dataset will remain available for reproducibility and comparison.
